In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

import json

In [2]:
df_train = pd.read_parquet("03_datasets/binned_train.parquet")
df_test = pd.read_parquet("03_datasets/binned_test.parquet")

In [3]:
df_train

,GameId,WhiteElo,BlackElo,Elo,Opening,ECO,FirstMoves,MeanStartLoss,MeanQueenLoss,MeanRookLoss,MeanKnightLoss,MeanBishopLoss,MeanKingLoss1,MeanKingLoss2,MeanPawnLoss1,MeanPawnLoss2,MeanFlagPawnLoss1,MeanFlagPawnLoss2,MeanCenterPawnLoss1,MeanCenterPawnLoss2
0,jgeqUmTV,1844,1819,1831,Scotch Game: Classical Variation,C45,e4 e5 Nf3 Nc6 d4 exd4 Nxd4 Bc5 Be3 d6 Nxc6 bxc...,0.000435,0.000000,0.000247,0.000251,0.000306,0.004690,0.000265,0.000641,0.000614,0.000643,0.000643,0.000596,0.000578
1,gdHoCJB5,1517,1504,1510,"Scotch Game: Classical Variation, Intermezzo V...",C45,e4 e5 Nf3 Nc6 d4 exd4 Nxd4 Bc5 Nxc6 Qf6 Be3 Bx...,0.000578,0.000684,0.000527,0.000449,0.000478,0.004690,0.000750,0.000825,0.000825,0.000000,0.000000,0.000929,0.000929
2,oVAuIb3u,2092,2096,2094,Indian Defense,A45,d4 Nf6 Bf4 g6 Nc3 Bg7 e4 d6 Qd2 c6 Bh6 Bxh6 Qx...,0.000592,0.000404,0.000761,0.001424,0.000679,0.019960,0.002040,0.000260,0.000281,0.000535,0.000383,0.000194,0.000226
3,cPMRdRO3,1684,1716,1700,Italian Game: Paris Defense,C50,e4 e5 Nf3 d6 Bc4 Nc6 Nc3 Be7 d4 f5 dxe5 Nxe5 N...,0.000368,0.000991,0.000430,0.000110,0.000302,0.012050,0.000608,0.000482,0.065285,0.000910,0.114672,0.000464,0.000464
4,dWmz6C3J,1653,1653,1653,Bishop's Opening: Berlin Defense,C24,e4 e5 Bc4 Nf6 d3 d5 Bb3 Bc5 Nc3 c6 Bg5 d4 Nce2...,0.000912,0.000079,0.000449,0.001499,0.000974,0.004690,0.001857,0.000377,0.000377,0.000221,0.000221,0.000480,0.000480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,R8gWFNvk,1857,1859,1858,Queen's Gambit Accepted,D20,d4 d5 c4 dxc4 Nc3 Nf6 e4 e6 Bxc4 Bb4 Qd3 O-O N...,0.021542,0.002058,0.237545,0.000654,0.000765,0.004589,0.000000,0.000706,0.000706,0.001062,0.001062,0.000609,0.000609
9996,LH3BRIWu,2154,2152,2153,Queen's Pawn Game: Accelerated London System,D00,d4 d5 Bf4 Nc6 e3 a6 Nd2 Bf5 c4 e6 Qb3 Rb8 cxd5...,0.000392,0.000136,0.000919,0.000519,0.000257,0.004589,0.000549,0.000478,0.000393,0.000357,0.000357,0.000413,0.000413
9997,SLD4S9fl,1429,1436,1432,Vienna Game: Vienna Gambit,C29,e4 e5 Nc3 Nf6 f4 d6 Nf3 g6 Bc4 Bg7 Ng5 O-O O-O...,0.000971,0.000119,0.001209,0.000614,0.000978,0.001428,0.000552,0.001526,0.005506,0.010375,0.007158,0.003474,0.001800
9998,za4cYSpL,1828,1834,1831,Dutch Defense,A80,d4 f5 Bf4 e6 e3 Nf6 Bd3 b6 Nf3 Bb7 O-O Be7 h3 ...,0.000399,0.000835,0.000346,0.000313,0.000457,0.004589,0.000013,0.000270,0.000256,0.000214,0.000214,0.000275,0.000275


In [4]:
df_train[["Opening", "ECO"]].head(10)

,Opening,ECO
0,Scotch Game: Classical Variation,C45
1,"Scotch Game: Classical Variation, Intermezzo V...",C45
2,Indian Defense,A45
3,Italian Game: Paris Defense,C50
4,Bishop's Opening: Berlin Defense,C24
5,Italian Game: Giuoco Pianissimo,C50
6,Nimzo-Larsen Attack: Modern Variation,A01
7,Russian Game: Three Knights Game,C42
8,Scandinavian Defense: Mieses-Kotroc Variation,B01
9,Benoni Defense: Old Benoni,A43


**Mean encoding**

In [5]:
df_train["OpeningType"] = df_train["Opening"].str.split(":", expand=True)[0]
df_train["ECOType"] = df_train["ECO"].str[:2]

df_test["OpeningType"] = df_test["Opening"].str.split(":", expand=True)[0]
df_test["ECOType"] = df_test["ECO"].str[:2]

In [6]:
# opening_features = ["ECO", "OpeningType", "ECOType", "Opening"]
opening_features = ["Opening"]
for feature in opening_features:
    
    # Common openings
    common_openings = (
        df_train[feature]
        .value_counts()
        .where(lambda x: x >= 100).dropna()
        .index
    )
    
    # Calculate the means
    means_dict = (
        df_train
        .groupby(feature)
        .agg({"Elo": "mean"})
        .loc[common_openings]
        .squeeze().to_dict()
    )
    
    # Mean for rare openings
    mean_if_rare = df_train["Elo"].where(
        ~df_train["Opening"].isin(common_openings)
    ).mean()
    
    # Apply
    df_train[feature] = (
        df_train[feature]
        .map(means_dict)
        .fillna(mean_if_rare)
    )
    
    df_test[feature] = (
        df_test[feature]
        .map(means_dict)
        .fillna(mean_if_rare)
    )

In [7]:
# s = pd.Series(means_dict).sort_values().reset_index(drop=True)

# fig = px.line(
#     s.values, 
#     template="plotly_white", 
# )
# fig.data[0].marker.line.width=0
# fig.update_xaxes(title="")
# fig.update_yaxes(title="")
# fig.update_layout(
#     height=1080//2, 
#     width=1920//2, 
#     font_size=20,
#     font_family="Consolas",
#     showlegend=False,
#     yaxis_range=[900, 2200]
# )

# fig.data[0].line.color="#008C45"
# fig.data[0].line.width=2

# fig.show()
# fig.write_image("presentation/images/opening_mean.png", scale=2)

**Opening line**

In [8]:
df_train["FirstMoves"] = df_train["FirstMoves"].str.split(" ")
df_test["FirstMoves"] = df_test["FirstMoves"].str.split(" ")

elos = df_train["Elo"].values
moves = df_train["FirstMoves"].values

In [9]:
line_count = {}
line_rating = {}

for elo, move in zip(elos, moves):
    
    for i in range(1, min(20, len(move))):
        
        key = " ".join(move[:i])
        line_count[key] = line_count.get(key, 0) + 1
        line_rating[key] = line_rating.get(key, 0) + elo

In [10]:
result = {
    k: v / line_count[k]
    for k, v in line_rating.items()
    if line_count[k] > 100
}

In [11]:
series = pd.Series(result).sort_values()

In [12]:

# fig = px.line(
#     series.values, 
#     template="plotly_white", 
# )
# fig.data[0].marker.line.width=0
# fig.update_xaxes(title="")
# fig.update_yaxes(title="")
# fig.update_layout(
#     height=1080//2, 
#     width=1920//2, 
#     font_size=20,
#     font_family="Consolas",
#     showlegend=False,
#     yaxis_range=[900, 2200]
# )

# fig.data[0].line.color="#008C45"
# fig.data[0].line.width=2

# fig.show()
# fig.write_image("presentation/images/line_mean.png", scale=2)

In [13]:
tree = {}
for key, value in series.sort_index().to_dict().items():
    current_node = tree
    for move in key.split(" "):
        if not (move in current_node):
            current_node[move] = {}
        current_node = current_node[move]
    current_node["mean"] = value

In [14]:
tree["e4"]["e5"]["Bc4"]["mean"]

1478.0767276422764

In [15]:
def get_line_mean(moves_list):
    try:
        current_node = tree
        for move in moves_list:
            if len(current_node) == 1:
                return current_node["mean"]
            if move in current_node:
                current_node = current_node[move]
            else:
                return current_node["mean"]
    except:
        return 1500

In [16]:
df_train["LineTreeMean"] = df_train["FirstMoves"].map(get_line_mean).fillna(1500)
df_test["LineTreeMean"] = df_test["FirstMoves"].map(get_line_mean).fillna(1500)

In [17]:
df_train.drop(columns=["FirstMoves"]).to_parquet("03_datasets/final_train.parquet")
df_test.drop(columns=["FirstMoves"]).to_parquet("03_datasets/final_test.parquet")